# Log File Analyzer

## Parse application log files to identify ERROR and WARNING entries, group them by module, and produce a CSV report showing error frequency.
### Expected Deliverables: Python script and generated CSV.



### Imports

In [1]:
import re
import pandas as pd
from collections import defaultdict

### Configure Input File

In [3]:
LOG_FILE = "./log_dataset/Hadoop_2k.log"
OUTPUT_CSV = "log_error_summary.csv"

### Parse Log File

In [4]:
log_pattern = re.compile(
    r"^(?P<timestamp>\d{4}-\d{2}-\d{2} "
    r"\d{2}:\d{2}:\d{2},\d+)\s+"
    r"(?P<level>INFO|WARN|ERROR|FATAL)\s+"
    r"\[.*?\]\s+"
    r"(?P<module>[^:]+):\s*"
    r"(?P<message>.*)$"
)

records = []

with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        match = log_pattern.match(line.strip())

        if match:
            records.append(match.groupdict())

print(f"Parsed {len(records)} log entries")

Parsed 2000 log entries


### Create DataFrame

In [5]:
df = pd.DataFrame(records)

print(df.shape)

df.head()

(2000, 4)


,timestamp,level,module,message
0,"2015-10-18 18:01:47,978",INFO,org.apache.hadoop.mapreduce.v2.app.MRAppMaster,Created MRAppMaster for application appattempt...
1,"2015-10-18 18:01:48,963",INFO,org.apache.hadoop.mapreduce.v2.app.MRAppMaster,Executing with tokens:
2,"2015-10-18 18:01:48,963",INFO,org.apache.hadoop.mapreduce.v2.app.MRAppMaster,"Kind: YARN_AM_RM_TOKEN, Service: , Ident: (app..."
3,"2015-10-18 18:01:49,228",INFO,org.apache.hadoop.mapreduce.v2.app.MRAppMaster,Using mapred newApiCommitter.
4,"2015-10-18 18:01:50,353",INFO,org.apache.hadoop.mapreduce.v2.app.MRAppMaster,OutputCommitter set in config null


### Extract ERROR and WARN Entries

In [8]:
print(df["level"].unique())

['INFO' 'ERROR' 'WARN' 'FATAL']


In [9]:
issues_df = df[
    df["level"].isin(["ERROR", "WARN", "FATAL"])
].copy()

print("Issue entries:", len(issues_df))

issues_df.head()

Issue entries: 960


,timestamp,level,module,message
667,"2015-10-18 18:04:11,034",ERROR,org.apache.hadoop.mapreduce.v2.app.rm.RMContai...,Container complete event for unknown container...
847,"2015-10-18 18:05:27,570",WARN,org.apache.hadoop.ipc.Client,Address change detected. Old: msra-sa-41/10.19...
848,"2015-10-18 18:05:27,570",WARN,org.apache.hadoop.hdfs.LeaseRenewer,Failed to renew lease for [DFSClient_NONMAPRED...
849,"2015-10-18 18:05:28,570",WARN,org.apache.hadoop.ipc.Client,Address change detected. Old: msra-sa-41/10.19...
850,"2015-10-18 18:05:28,570",WARN,org.apache.hadoop.hdfs.LeaseRenewer,Failed to renew lease for [DFSClient_NONMAPRED...


### Count Errors by Module

In [10]:
error_summary = (
    issues_df
    .groupby(["module", "level"])
    .size()
    .unstack(fill_value=0)
)

error_summary

level,ERROR,FATAL,WARN
module,,,
org.apache.hadoop.hdfs.DFSClient,0,0,4
org.apache.hadoop.hdfs.LeaseRenewer,0,0,326
org.apache.hadoop.ipc.Client,0,0,476
org.apache.hadoop.mapred.TaskAttemptListenerImpl,0,2,0
org.apache.hadoop.mapreduce.jobhistory.JobHistoryEventHandler,1,0,0
org.apache.hadoop.mapreduce.v2.app.commit.CommitterEventHandler,0,0,2
org.apache.hadoop.mapreduce.v2.app.rm.RMContainerAllocator,148,0,0
org.apache.hadoop.yarn.YarnUncaughtExceptionHandler,1,0,0


### Total Error Frequency

In [11]:
error_summary["TOTAL_ISSUES"] = error_summary.sum(axis=1)

error_summary = (
    error_summary
    .sort_values(
        "TOTAL_ISSUES",
        ascending=False
    )
)

error_summary.head(20)

level,ERROR,FATAL,WARN,TOTAL_ISSUES
module,,,,
org.apache.hadoop.ipc.Client,0,0,476,476
org.apache.hadoop.hdfs.LeaseRenewer,0,0,326,326
org.apache.hadoop.mapreduce.v2.app.rm.RMContainerAllocator,148,0,0,148
org.apache.hadoop.hdfs.DFSClient,0,0,4,4
org.apache.hadoop.mapred.TaskAttemptListenerImpl,0,2,0,2
org.apache.hadoop.mapreduce.v2.app.commit.CommitterEventHandler,0,0,2,2
org.apache.hadoop.mapreduce.jobhistory.JobHistoryEventHandler,1,0,0,1
org.apache.hadoop.yarn.YarnUncaughtExceptionHandler,1,0,0,1


### Save CSV Report

In [12]:
error_summary.to_csv(OUTPUT_CSV)

print(
    f"CSV report saved as: {OUTPUT_CSV}"
)

CSV report saved as: log_error_summary.csv


In [13]:
print("Total Log Entries:", len(df))

print(
    "Total Error/Warning/Fatal Entries:",
    len(issues_df)
)

print("\nLog Level Distribution")

print(
    issues_df["level"]
    .value_counts()
)

Total Log Entries: 2000
Total Error/Warning/Fatal Entries: 960

Log Level Distribution
level
WARN     808
ERROR    150
FATAL      2
Name: count, dtype: int64


### Top Problematic Modules

In [14]:
top_modules = (
    error_summary["TOTAL_ISSUES"]
    .head(10)
)

top_modules

module
org.apache.hadoop.ipc.Client                                       476
org.apache.hadoop.hdfs.LeaseRenewer                                326
org.apache.hadoop.mapreduce.v2.app.rm.RMContainerAllocator         148
org.apache.hadoop.hdfs.DFSClient                                     4
org.apache.hadoop.mapred.TaskAttemptListenerImpl                     2
org.apache.hadoop.mapreduce.v2.app.commit.CommitterEventHandler      2
org.apache.hadoop.mapreduce.jobhistory.JobHistoryEventHandler        1
org.apache.hadoop.yarn.YarnUncaughtExceptionHandler                  1
Name: TOTAL_ISSUES, dtype: int64